In [1]:
import os
import json
import math
import pandas as pd
from PIL import Image

In [2]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

BASE_DIR = os.path.dirname(os.path.abspath("*"))

GT_FOLDER   = os.path.join(BASE_DIR,"clicking_mechanism", "annotations")
PRED_FOLDER = os.path.join(BASE_DIR, "annotated_output", "json")
IMAGE_FOLDER = os.path.join(BASE_DIR, "data")

In [3]:
def display_first_json_as_df(folder_path, folder_name):
    json_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".json")])
    
    if not json_files:
        print(f"No JSON files found in {folder_name}")
        return
    
    first_file = json_files[0]
    full_path = os.path.join(folder_path, first_file)

    print(f"\n===== First JSON in {folder_name}: {first_file} =====")
    
    with open(full_path, "r") as f:
        data = json.load(f)

    # If JSON is a list of annotations
    if isinstance(data, list):
        df = pd.json_normalize(data)

    # If JSON is a dictionary with nested items
    elif isinstance(data, dict):
        # Try flattening everything
        df = pd.json_normalize(data)

    else:
        print("Unknown JSON structure")
        return
    
    display(df)   # Jupyter display
    return df


# Display both
gt_df = display_first_json_as_df(GT_FOLDER, "Ground Truth Folder")
pred_df = display_first_json_as_df(PRED_FOLDER, "Prediction Folder")


===== First JSON in Ground Truth Folder: form1.json =====


,checkboxes,lines,boxes
0,"[[537.5, 478.12], [759.38, 478.12], [987.5, 47...","[[262.5, 1365.62, 612.5, 1368.75], [300.0, 144...",[]



===== First JSON in Prediction Folder: form1.json =====


,checkboxes,input_boxes,lines,meta.source_file,meta.image_size
0,"[[985.5, 483.5], [756.5, 483.5], [542.5, 483.5]]",[],"[[1073.0, 1898.5, 1603.0, 1898.5], [1505.0, 17...",form1.png,"[1700, 2200]"


In [4]:
#Distance-Based Best Matching (Order Independent)

In [5]:
TOLERANCES = [0.05, 0.10, 0.20]


# ─────────────────────────────────────────────
# UTIL FUNCTIONS
# ─────────────────────────────────────────────

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


# ─────────────────────────────────────────────
# ORDER-INDEPENDENT MATCHING (Closest First)
# ─────────────────────────────────────────────

def match_points(gt_points, pred_points, tol_x, tol_y):
    matches = []

    for gi, (gx, gy) in enumerate(gt_points):
        for pi, (px, py) in enumerate(pred_points):
            if abs(px - gx) <= tol_x and abs(py - gy) <= tol_y:
                dist = math.sqrt((gx - px)**2 + (gy - py)**2)
                matches.append((dist, gi, pi))

    matches.sort(key=lambda x: x[0])

    matched_gt = set()
    matched_pred = set()
    final_matches = 0

    for _, gi, pi in matches:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi)
            matched_pred.add(pi)
            final_matches += 1

    return final_matches


def match_lines(gt_lines, pred_lines, tol_x, tol_y):
    matches = []

    for gi, g in enumerate(gt_lines):
        gx1, gy1, gx2, gy2 = g

        for pi, p in enumerate(pred_lines):
            px1, py1, px2, py2 = p

            if (abs(px1 - gx1) <= tol_x and
                abs(py1 - gy1) <= tol_y and
                abs(px2 - gx2) <= tol_x and
                abs(py2 - gy2) <= tol_y):

                dist = (
                    math.sqrt((gx1 - px1)**2 + (gy1 - py1)**2) +
                    math.sqrt((gx2 - px2)**2 + (gy2 - py2)**2)
                )

                matches.append((dist, gi, pi))

    matches.sort(key=lambda x: x[0])

    matched_gt = set()
    matched_pred = set()
    final_matches = 0

    for _, gi, pi in matches:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi)
            matched_pred.add(pi)
            final_matches += 1

    return final_matches


def match_boxes(gt_boxes, pred_boxes, tol_x, tol_y):
    matches = []

    for gi, g in enumerate(gt_boxes):
        gx1, gy1, gx2, gy2 = g

        for pi, p in enumerate(pred_boxes):
            px1, py1, px2, py2 = p

            if (abs(px1 - gx1) <= tol_x and
                abs(py1 - gy1) <= tol_y and
                abs(px2 - gx2) <= tol_x and
                abs(py2 - gy2) <= tol_y):

                dist = (
                    math.sqrt((gx1 - px1)**2 + (gy1 - py1)**2) +
                    math.sqrt((gx2 - px2)**2 + (gy2 - py2)**2)
                )

                matches.append((dist, gi, pi))

    matches.sort(key=lambda x: x[0])

    matched_gt = set()
    matched_pred = set()
    final_matches = 0

    for _, gi, pi in matches:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi)
            matched_pred.add(pi)
            final_matches += 1

    return final_matches


# ─────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────

def compute_metrics(matches, total_pred, total_gt):
    precision = matches / total_pred if total_pred else 0
    recall = matches / total_gt if total_gt else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0
    return precision, recall, f1


# ─────────────────────────────────────────────
# MAIN EVALUATION
# ─────────────────────────────────────────────

def evaluate():

    files = [f for f in os.listdir(GT_FOLDER) if f.endswith(".json")]

    for tol in TOLERANCES:

        total_gt_cb = total_pred_cb = total_match_cb = 0
        total_gt_ln = total_pred_ln = total_match_ln = 0
        total_gt_box = total_pred_box = total_match_box = 0

        print("\n" + "="*50)
        print(f"TOLERANCE: {int(tol*100)}%")
        print("="*50)

        for file in files:

            gt_path = os.path.join(GT_FOLDER, file)
            pred_path = os.path.join(PRED_FOLDER, file)

            if not os.path.exists(pred_path):
                continue

            image_name = file.replace(".json", ".png")
            image_path = os.path.join(IMAGE_FOLDER, image_name)

            if not os.path.exists(image_path):
                continue

            width, height = Image.open(image_path).size
            tol_x = width * tol
            tol_y = height * tol

            gt_data = load_json(gt_path)
            pred_data = load_json(pred_path)

            gt_cb = gt_data.get("checkboxes", [])
            pred_cb = pred_data.get("checkboxes", [])

            gt_ln = gt_data.get("lines", [])
            pred_ln = pred_data.get("lines", [])

            gt_box = gt_data.get("boxes", [])
            pred_box = pred_data.get("input_boxes", [])

            match_cb = match_points(gt_cb, pred_cb, tol_x, tol_y)
            match_ln = match_lines(gt_ln, pred_ln, tol_x, tol_y)
            match_box = match_boxes(gt_box, pred_box, tol_x, tol_y)

            total_gt_cb += len(gt_cb)
            total_pred_cb += len(pred_cb)
            total_match_cb += match_cb

            total_gt_ln += len(gt_ln)
            total_pred_ln += len(pred_ln)
            total_match_ln += match_ln

            total_gt_box += len(gt_box)
            total_pred_box += len(pred_box)
            total_match_box += match_box

        # Compute Metrics
        cb_p, cb_r, cb_f1 = compute_metrics(total_match_cb, total_pred_cb, total_gt_cb)
        ln_p, ln_r, ln_f1 = compute_metrics(total_match_ln, total_pred_ln, total_gt_ln)
        box_p, box_r, box_f1 = compute_metrics(total_match_box, total_pred_box, total_gt_box)

        print("\nCheckboxes:")
        print(f"Precision: {cb_p:.3f} | Recall: {cb_r:.3f} | F1: {cb_f1:.3f}")

        print("\nLines:")
        print(f"Precision: {ln_p:.3f} | Recall: {ln_r:.3f} | F1: {ln_f1:.3f}")

        print("\nBoxes:")
        print(f"Precision: {box_p:.3f} | Recall: {box_r:.3f} | F1: {box_f1:.3f}")


if __name__ == "__main__":
    evaluate()


TOLERANCE: 5%

Checkboxes:
Precision: 0.817 | Recall: 0.683 | F1: 0.744

Lines:
Precision: 0.215 | Recall: 0.453 | F1: 0.291

Boxes:
Precision: 0.273 | Recall: 0.590 | F1: 0.374

TOLERANCE: 10%

Checkboxes:
Precision: 0.819 | Recall: 0.685 | F1: 0.746

Lines:
Precision: 0.228 | Recall: 0.480 | F1: 0.309

Boxes:
Precision: 0.295 | Recall: 0.638 | F1: 0.404

TOLERANCE: 20%

Checkboxes:
Precision: 0.822 | Recall: 0.687 | F1: 0.749

Lines:
Precision: 0.251 | Recall: 0.530 | F1: 0.341

Boxes:
Precision: 0.347 | Recall: 0.750 | F1: 0.474
